Rules for universe selection:

1. Listed on BME main market (Mercado Continuo) and currently trading.
2. Market cap below €1bn **and** not on the 2026 transaction-tax list, so no 0.2% tax on purchases.
3. At least 10 years of daily price history, so every stock covers the full backtest window.
5. Not under a takeover, in insolvency, or in a court restructuring.
6. Top 10 of the remaining companies with the highest median daily volume over the last 12 months.

In [3]:
from pathlib import Path

import pandas as pd
import yfinance as yf
import warnings
ROOT = Path.cwd().parent
REF = ROOT / "data" / "reference"
OUT = ROOT / "data" / "universe"
OUT.mkdir(parents=True, exist_ok=True)

AS_OF = "2026-10-23"                       # last trading day before the selection
CAP_LIMIT = 1_000_000_000                  # €1bn
MIN_FIRST_DATE = pd.Timestamp("2016-10-24")  # 10 years of history
N_STOCKS = 10

In [4]:
mc = pd.read_csv(REF / "mercado_continuo_2026-10-23.csv")
print(len(mc), "stocks on the Mercado Continuo")
mc.head()

123 stocks on the Mercado Continuo


,bme_name,yahoo_symbol,status,suspended_since,domicile
0,ACCIONA,ANA.MC,active,NaN,ES
1,ACCIONA ENER,ANE.MC,active,NaN,ES
2,ACERINOX,ACX.MC,active,NaN,ES
3,ACS,ACS.MC,active,NaN,ES
4,ADOLFO DGUEZ,ADZ.MC,active,NaN,ES


In [5]:
mc["status"].value_counts()

status
active       118
suspended      5
Name: count, dtype: int64

In [6]:
snap_path = OUT / f"yahoo_market_caps_{AS_OF}.csv"

if snap_path.exists():
    caps = pd.read_csv(snap_path)
else:
    res = yf.screen(yf.EquityQuery("eq", ["exchange", "MCE"]), size=250)
    caps = pd.DataFrame(res["quotes"])[["symbol", "marketCap"]]
    caps = caps.rename(columns={"symbol": "yahoo_symbol", "marketCap": "market_cap"})
    caps.to_csv(snap_path, index=False)

# a left merge keeps every row of mc and attaches the matching market cap
universe = mc.merge(caps, on="yahoo_symbol", how="left")
universe[["bme_name", "yahoo_symbol", "market_cap"]].head()

,bme_name,yahoo_symbol,market_cap
0,ACCIONA,ANA.MC,1.092820e+10
1,ACCIONA ENER,ANE.MC,6.984304e+09
2,ACERINOX,ACX.MC,4.146001e+09
3,ACS,ACS.MC,2.425992e+10
4,ADOLFO DGUEZ,ADZ.MC,6.335713e+07


In [7]:
ftt = pd.read_csv(REF / "ftt_list_2026.csv")

is_active = universe["status"] == "active"
is_spanish = universe["domicile"] == "ES"
is_small = universe["market_cap"] < CAP_LIMIT
not_taxed = ~universe["yahoo_symbol"].isin(ftt["yahoo_symbol"])

candidates = universe[is_active & is_spanish & is_small & not_taxed].copy()
print(len(universe), "->", len(candidates), "candidates")

123 -> 49 candidates


Aedas Homes is worth less than €1bn today, but it was above €1bn on 1 Dec 2025, so its purchases are taxed for all of 2026.

In [8]:
universe[is_small & ~not_taxed][["bme_name", "market_cap"]]

,bme_name,market_cap
5,AEDAS,937066496.0


We use `auto_adjust=False` to get the real price on the day. Adjusted prices are rescaled for dividends and splits, which is right for measuring returns. To measure volume, we need the real price on the day.

In [9]:
hist = yf.download(candidates["yahoo_symbol"].tolist(), period="max", auto_adjust=False, progress=False)

hist["Close"].iloc[-3:, :5]

C:\Users\mayboll\Desktop\strats\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\mayboll\Desktop\strats\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\mayboll\Desktop\strats\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=T

Ticker,ADX.MC,ADZ.MC,AI.MC,ALNT.MC,AMP.MC
Date,,,,,
2026-10-21,1.060,6.95,0.0835,8.94,4.400
2026-10-22,1.032,7.00,0.0831,8.84,4.345
2026-10-23,1.028,7.00,0.0823,8.76,4.460


In [10]:
# first date with a price, per ticker
first_date = hist["Close"].apply(lambda col: col.first_valid_index())

# last 12 months only
last_year = hist.loc["2025-10-24":AS_OF]

# two tables with the same rows and columns multiply cell by cell: price x shares = euros traded
euro_volume = last_year["Close"] * last_year["Volume"]

liquidity = pd.DataFrame({
    "median_eur_volume": euro_volume.median(),
    "mean_eur_volume": euro_volume.mean(),
    "zero_volume_days": (last_year["Volume"] == 0).sum(),
    "first_date": first_date,
})

Notice below that the mean is above the median for every stock: daily volume is skewed, with a few very large days. Therefore we will choose the median.

In [11]:
exclusions = {
    "SOL.MC": "suspended Sep 2024 to Dec 2025 during a court restructuring; new majority owner",
}

In [12]:
table = candidates.set_index("yahoo_symbol")[["bme_name", "market_cap"]].join(liquidity)

table["reason_out"] = pd.Series(exclusions)
too_short = table["first_date"] > MIN_FIRST_DATE
table.loc[too_short & table["reason_out"].isna(), "reason_out"] = "less than 10 years of history"

table = table.sort_values("median_eur_volume", ascending=False)
table.to_csv(OUT / f"selection_table_{AS_OF}.csv")

pd.options.display.float_format = "{:,.0f}".format
table.head(15)

,bme_name,market_cap,median_eur_volume,mean_eur_volume,zero_volume_days,first_date,reason_out
yahoo_symbol,,,,,,,
OHLA.MC,OHLA,"456,207,872","2,771,039","3,740,987",0,2000-01-03,NaN
EDR.MC,EDREAMS,"473,121,536","2,308,550","3,022,513",0,2014-04-08,NaN
AMP.MC,AMPER,"440,189,216","1,362,706","1,979,836",0,2000-01-03,NaN
TSK.MC,TSK,"707,207,552","1,070,154","1,585,064",0,2026-05-13,less than 10 years of history
ORY.MC,ORYZON,"204,893,728","910,657","1,209,757",0,2015-12-14,NaN
TUB.MC,TUBACEX,"338,412,160","870,310","1,042,578",0,2000-01-03,NaN
ENC.MC,ENCE,"534,140,928","708,032","858,756",0,2002-01-02,NaN
NXT.MC,NEXTIL,"435,829,344","686,127","924,979",0,2000-01-03,NaN
DOM.MC,DOMINION,"384,623,936","684,337","850,956",0,2016-04-27,NaN


In [13]:
selected = table[table["reason_out"].isna()].head(N_STOCKS)
selected.to_csv(OUT / "universe_v1.csv")
selected[["bme_name", "market_cap", "median_eur_volume", "first_date"]]

,bme_name,market_cap,median_eur_volume,first_date
yahoo_symbol,,,,
OHLA.MC,OHLA,"456,207,872","2,771,039",2000-01-03
EDR.MC,EDREAMS,"473,121,536","2,308,550",2014-04-08
AMP.MC,AMPER,"440,189,216","1,362,706",2000-01-03
ORY.MC,ORYZON,"204,893,728","910,657",2015-12-14
TUB.MC,TUBACEX,"338,412,160","870,310",2000-01-03
ENC.MC,ENCE,"534,140,928","708,032",2002-01-02
NXT.MC,NEXTIL,"435,829,344","686,127",2000-01-03
DOM.MC,DOMINION,"384,623,936","684,337",2016-04-27
GSJ.MC,SAN JOSE,"448,029,696","409,552",2009-07-20


## Known limitations (these go in the write-up)

- **Survivorship bias.** The stocks were chosen by today's size and status, but they'll be tested on the last 10 years. A company that is small today may have been much bigger in 2017, and it could only become small by falling. Free data has no historical lists of which stocks were small at the time, so we state the bias and check that results aren't driven by one or two names.
- **Volume is BME only.** Yahoo's volume for `.MC` tickers is trading on the Madrid exchange. Some trading in Spanish stocks also happens on other European venues, so the true volume is higher. That's fine for *ranking*, but don't read the numbers as total liquidity.
- **The tax list changes each year.** This selection uses the 2026 list. The backtest has to charge 0.2% on purchases in any year a stock was on that year's list (the tax started in January 2021).